In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [15]:

## Load Cleaned Data
df = pd.read_csv('../data/processed/student_placement_cleaned.csv')

In [16]:
list(df.columns)

['student_id',
 'cgpa',
 'branch',
 'college_tier',
 'python_skill',
 'dsa_skill',
 'ml_skill',
 'web_dev_skill',
 'coding_score',
 'communication_score',
 'aptitude_score',
 'internships',
 'projects',
 'backlogs',
 'resume_score',
 'skill_score',
 'placed',
 'company_type',
 'job_role',
 'salary_lpa',
 'placement_status']

In [17]:
# We use the median as the cutoff for "High" vs "Low"
cgpa_median = df['cgpa'].median()
skill_median = df['skill_score'].median()

def create_segment(row):
    if row['skill_score'] > skill_median and row['cgpa'] > cgpa_median:
        return '1. High Skill + High CGPA'
    elif row['skill_score'] > skill_median and row['cgpa'] <= cgpa_median:
        return '2. High Skill + Low CGPA'
    elif row['skill_score'] <= skill_median and row['cgpa'] > cgpa_median:
        return '3. Low Skill + High CGPA'
    else:
        return '4. Low Skill + Low CGPA'

# Apply the segmentation
df['student_segment'] = df.apply(create_segment, axis=1)

In [18]:
df.head()

,student_id,cgpa,branch,college_tier,python_skill,dsa_skill,ml_skill,web_dev_skill,coding_score,communication_score,...,projects,backlogs,resume_score,skill_score,placed,company_type,job_role,salary_lpa,placement_status,student_segment
0,S0,6.87,Civil,1,1,1,0,0,15.6,4.3,...,3,0,62.6,2,1,MNC,Software Engineer,63.55,Placed,4. Low Skill + Low CGPA
1,S1,6.52,Civil,2,1,0,0,1,13.9,5.8,...,6,0,77.5,2,1,MNC,Data Scientist,75.17,Placed,4. Low Skill + Low CGPA
2,S2,5.33,IT,1,1,1,1,0,9.8,8.1,...,5,1,76.0,3,1,MNC,Software Engineer,80.44,Placed,2. High Skill + Low CGPA
3,S3,6.04,Civil,3,1,0,1,0,39.5,9.6,...,6,0,74.3,2,1,MNC,Software Engineer,72.11,Placed,4. Low Skill + Low CGPA
4,S4,6.78,Mechanical,2,0,1,0,1,7.5,9.9,...,3,0,66.8,2,1,Mid-size,Software Engineer,67.05,Placed,4. Low Skill + Low CGPA


### Feature Creation

In [19]:
# Creat an Exprience Score
df["experience_score"] = (
    (df["internships"] * 0.6) +
    (df["projects"] * 0.4)
)

In [20]:
## Binary backlog flag
df['has_backlog'] = np.where(df['backlogs'] > 0, 1, 0)

In [21]:
## Academic Risk Flag
df["academic_risk"] = df["has_backlog"].apply(
    lambda x: "High Risk" if x > 0 else "Low Risk"
)

In [22]:
def salary_category(x):
    if x == 0:
        return "Not Placed"
    elif x <= 5:
        return "Low"
    elif x <= 10:
        return "Medium"
    else:
        return "High"

df["salary_tier"] = df["salary_lpa"].apply(salary_category)

In [23]:
df["readiness_score"] = (
    0.30 * df["skill_score"] +
    0.20 * df["coding_score"] +
    0.15 * df["internships"] +
    0.15 * df["projects"] +
    0.10 * df["communication_score"] +
    0.10 * df["aptitude_score"]
)

In [24]:
# Select only numeric columns and drop ID + Salary (to prevent data leakage)
numeric_df = df.select_dtypes(include=['number']).drop(columns=['student_id', 'salary_lpa'], errors='ignore')

# Calculate the correlation matrix
corr_matrix = numeric_df.corr()

# Print the exact rankings for easy reading
print("--- Top Factors Influencing PLACEMENT ---")
print(corr_matrix['placed'].sort_values(ascending=False))
print("\n")


--- Top Factors Influencing PLACEMENT ---
placed                 1.000000
resume_score           0.394616
skill_score            0.282007
experience_score       0.262111
readiness_score        0.248855
coding_score           0.207325
internships            0.196917
projects               0.176098
dsa_skill              0.147735
python_skill           0.143134
web_dev_skill          0.140937
ml_skill               0.127094
communication_score    0.119618
cgpa                   0.095778
aptitude_score         0.076450
college_tier          -0.207083
has_backlog           -0.253706
backlogs              -0.303126
Name: placed, dtype: float64




In [27]:
new_features = [
    "experience_score",
    "academic_risk",
    "readiness_score",
    "salary_tier",
    "student_segment"
]

df[new_features].head()

,experience_score,academic_risk,readiness_score,salary_tier,student_segment
0,1.8,Low Risk,13.95,High,4. Low Skill + Low CGPA
1,3.0,Low Risk,11.21,High,4. Low Skill + Low CGPA
2,2.0,High Risk,11.06,High,2. High Skill + Low CGPA
3,2.4,Low Risk,18.72,High,4. Low Skill + Low CGPA
4,1.2,Low Risk,12.17,High,4. Low Skill + Low CGPA


### Feature Engineering Observations

1. Resume score remains the strongest positive factor associated with placement outcomes.

2. Experience_score (internships + projects) demonstrated stronger association with placement than its individual components, suggesting combined practical exposure is more informative.

3. Skill-related metrics consistently appear among the top placement drivers.

4. Backlogs continue to show the strongest negative relationship with placement probability.

5. Readiness_score introduced additional signal and may be useful for business interpretation and dashboard insights.

In [25]:
# Save the feature-engineered dataset to the processed data folder
output_path = '../data/processed/student_placement_features.csv'
df.to_csv(output_path, index=False)
print(f"✓ Feature-engineered dataset saved successfully to: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nNew Features Created:")
print(f"  - student_segment: Skill + CGPA segmentation")
print(f"  - experience_score: Weighted internships & projects")
print(f"  - has_backlog: Binary backlog flag")
print(f"  - academic_risk: Risk categorization")
print(f"  - salary_tier: Salary brackets")
print(f"  - readiness_score: Overall placement readiness")
print(f"\nTotal columns: {df.shape[1]}")
print(f"Columns: {list(df.columns)}")

✓ Feature-engineered dataset saved successfully to: ../data/processed/student_placement_features.csv
Shape: (9000, 27)

New Features Created:
  - student_segment: Skill + CGPA segmentation
  - experience_score: Weighted internships & projects
  - has_backlog: Binary backlog flag
  - academic_risk: Risk categorization
  - salary_tier: Salary brackets
  - readiness_score: Overall placement readiness

Total columns: 27
Columns: ['student_id', 'cgpa', 'branch', 'college_tier', 'python_skill', 'dsa_skill', 'ml_skill', 'web_dev_skill', 'coding_score', 'communication_score', 'aptitude_score', 'internships', 'projects', 'backlogs', 'resume_score', 'skill_score', 'placed', 'company_type', 'job_role', 'salary_lpa', 'placement_status', 'student_segment', 'experience_score', 'has_backlog', 'academic_risk', 'salary_tier', 'readiness_score']
